# CPE15 MIDTERM PROJECT - Cleaning Notebook
---
## Part I: Field Data Collection, Validation, Cleaning, and Documentation
---
### BSCPE 3GF
### Submitted on: Oct 11, 2026




In [344]:
import re

import numpy as np
import pandas as pd

## Function Definitions

In [345]:
# Canonical schema shared by every raw CSV, plus a loader that enforces it
# The 16 required columns, in the order required by the spec
columns = [
    'feature_id', 'name', 'category', 'subcategory',
    'street', 'barangay', 'latitude', 'longitude',
    'location_description', 'operating_status', 'opening_hours',
    'contact_info', 'date_collected', 'collector',
    'remarks', 'verified'
]

# SW Maps GPS/telemetry columns, kept for the QA checks in section 10
# instead of being dropped at load time
qa_columns = [
    'Time', 'Horizontal Accuracy', 'HDOP',
    'Satellites in Use', 'Fix ID', 'Averaged Count'
]

# Non-spec raw headers that are merged into the schema or kept for traceability
# (Latitude/Longitude are renamed; Remarks is merged; Image/establishment_pic
# hold the photo reference in files 9 and 10)
raw_only_columns = ['Latitude', 'Longitude', 'Remarks', 'Image', 'establishment_pic']

# Everything the loader is allowed to pull out of each raw CSV
read_columns = set(columns) | set(qa_columns) | set(raw_only_columns)

# Final column order of the merged raw DataFrame
output_columns = columns + ['source_file', 'establishment_pic'] + qa_columns


def read_csv_files(x):
  """Read one raw CSV as text and return the spec columns, GPS QA columns, and traceability columns."""

  # dtype=str keeps phone numbers, leading zeros, and IDs exactly as written
  # (without it, e.g. 09602081305 in files 2 and 4 becomes 9602081305.0)
  df = pd.read_csv(x, dtype=str, usecols=lambda c: c in read_columns)

  # SW Maps exports use capitalized coordinate headers
  df = df.rename(columns={'Latitude': 'latitude', 'Longitude': 'longitude'})

  # Merge the stray capitalized `Remarks` column (its single value already
  # appears in `remarks` with different casing) into `remarks`
  if 'Remarks' in df.columns:
    df['remarks'] = df['remarks'].fillna(df.pop('Remarks'))

  # Files 9 and 10 store the photo reference under different headers
  # (`Image` vs `establishment_pic`); keep one `establishment_pic` column
  if 'establishment_pic' not in df.columns:
    df['establishment_pic'] = pd.NA
  if 'Image' in df.columns:
    df['establishment_pic'] = df['establishment_pic'].fillna(df['Image'])
    df = df.drop(columns='Image')

  return df[columns + ['establishment_pic'] + qa_columns]

In [346]:
# Parse the mixed-format date_collected column into proper datetime values
def clean_mixed_dates(df):
    """Clean and parse date_collected in place, then return the DataFrame"""

    # Cast to string, strip whitespace, and replace underscores with hyphens
    df['date_collected'] = (
        df['date_collected']
        .astype('string')
        .str.strip()
        .str.replace('_', '-', regex=False)
    )

    # Parse day-first mixed formats; unparseable values are coerced to NaT
    df['date_collected'] = pd.to_datetime(
        df['date_collected'],
        format='mixed',
        dayfirst=True,
        errors='coerce'
    )

    return df

In [347]:
# Canonical feature_id pattern required by the spec (section 5)
FEATURE_ID_PATTERN = r'^LUC_B\d{2}_\d{3}$'


def standardize_feature_id(fid):
    """Repair one feature_id to the canonical LUC_BNN_NNN form.

    Rule-based fixes replace the hard-coded dict: stray spaces,
    hyphens, the LIC_ prefix typo, letter O used for digit 0,
    and sequence numbers shorter than 3 digits.
    """
    s = str(fid).strip()

    # Normalize separators: drop all spaces, hyphens become underscores
    s = re.sub(r'\s+', '', s).replace('-', '_')

    # Fix the observed prefix typo (LIC_ written instead of LUC_)
    s = re.sub(r'^LIC_', 'LUC_', s)

    # Letter O -> digit 0; a valid ID never contains the letter O
    s = s.replace('O', '0')

    # Zero-pad the sequence part to 3 digits (LUC_B04_20 -> LUC_B04_020)
    m = re.fullmatch(r'(LUC_B\d{2})_(\d+)', s)
    if m:
        s = f'{m.group(1)}_{int(m.group(2)):03d}'

    return s

## Importing the raw datas (csv files from Google Drive)

In [348]:
# Load every raw CSV, clean its dates, tag its source, and collect the results
# uc?export=download&id=

# (Reference) Original Google Drive download URL mapping, now commented out
"""raw_datasets = { # raw datasets and their respective Google Drive URL links dictionary
    'raw_dataset_1': 'https://drive.google.com/uc?export=download&id=162Obps98STe5tsq5bFEl3gAzDPxfqblR',
    'raw_dataset_2': 'https://drive.google.com/uc?export=download&id=1QLZKJY8HiF-IQvFVZbpia9qA2yroEmoS',
    'raw_dataset_3': 'https://drive.google.com/uc?export=download&id=1RXOvapKILTYGc1biL42rjFjlB-I5t9JM',
    'raw_dataset_4': 'https://drive.google.com/uc?export=download&id=1Kze7v7q5nCR0nWUjlPjsjTKUtvWGibDL',
    'raw_dataset_5': 'https://drive.google.com/uc?export=download&id=196zcPBcRJIJgNMhmgRsrgJLwpC5gEZD5',
    'raw_dataset_6': 'https://drive.google.com/uc?export=download&id=1o0KullNbJ5t0cE_fm3QFSagFLfIFmrpC',
    'raw_dataset_7': 'https://drive.google.com/uc?export=download&id=1lTc9XVU5lgTNG0dGgKnnGnw3TGQJjq0S',
    'raw_dataset_8': 'https://drive.google.com/uc?export=download&id=1Celn9JNcsmKjmjjsxSo9BwW0BxSzspn1',
    'raw_dataset_9': 'https://drive.google.com/uc?export=download&id=1-_jGqHAioiHgjfKxUuWmuyV2ebja9Qoc',
    'raw_dataset_10': 'https://drive.google.com/uc?export=download&id=1KEqPvlOY4M43ptcfC2Qmg5CKpDb9ravw'
}"""

raw_datasets = { # raw datasets and their respective Google Drive URL links dictionary
    'raw_dataset_1': 'csv/raw_dataset_1.csv',
    'raw_dataset_2': 'csv/raw_dataset_2.csv',
    'raw_dataset_3': 'csv/raw_dataset_3.csv',
    'raw_dataset_4': 'csv/raw_dataset_4.csv',
    'raw_dataset_5': 'csv/raw_dataset_5.csv',
    'raw_dataset_6': 'csv/raw_dataset_6.csv',
    'raw_dataset_7': 'csv/raw_dataset_7.csv',
    'raw_dataset_8': 'csv/raw_dataset_8.csv',
    'raw_dataset_9': 'csv/raw_dataset_9.csv',
    'raw_dataset_10': 'csv/raw_dataset_10.csv'
}

# Holds the cleaned DataFrame for each raw dataset, keyed by dataset name
processed_datasets = {}

# For each raw CSV: read it, clean its dates, tag it with its source file name, store it
for name, url in raw_datasets.items():
    df = read_csv_files(url)
    df = clean_mixed_dates(df)
    df['source_file'] = name
    processed_datasets[name] = df

### Checking individual datasets

In [349]:
# Preview the first 5 rows of raw_dataset_1 to spot-check the load
df1 = processed_datasets['raw_dataset_1']
df1.head()

,feature_id,name,category,subcategory,street,barangay,latitude,longitude,location_description,operating_status,...,remarks,verified,establishment_pic,Time,Horizontal Accuracy,HDOP,Satellites in Use,Fix ID,Averaged Count,source_file
0,LUC_B10_001,"mommy's tacos, burgers and pizzas",Food and Dining,Eatery,G.Cadeliña,Barangay 10,14.112738333,121.55746,food stall at the front with an eatery inside,open,...,NaN,Yes,<NA>,10/03/2026 09:32:59.000 GMT+08:00,2.0,0.53,32.0,1.0,11.0,raw_dataset_1
1,LUC_B10_002,bnb laundry,Personal Services,Laundry,G.Cadeliña,Barangay 10,14.112738333,121.55746,"Next to mommy's tacos, burgers and pizzas",open,...,NaN,Yes,<NA>,10/03/2026 09:34:53.000 GMT+08:00,2.1,0.5,41.0,1.0,114.0,raw_dataset_1
2,LUC_B10_003,Star commecial,Retail,Grocery,G.Cadeliña,Barangay 10,14.112719583,121.557560667,"face to face of mommy's tacos, burger and pizzas",open,...,NaN,Yes,<NA>,10/03/2026 09:37:45.000 GMT+08:00,3.7,99.99,44.0,1.0,20.0,raw_dataset_1
3,LUC_B10_004,ICE ICE BABY,Other,Other,G.Cadeliña,Barangay 10,14.112879561,121.55752386,front have a gate colored brown,open,...,NaN,Yes,<NA>,10/03/2026 09:43:21.000 GMT+08:00,1.8,0.51,41.0,1.0,19.0,raw_dataset_1
4,LUC_B10_005,Zendong,Food and Dining,Eatery,G.Cadeliña,Barangay 10,14.11294,121.557613333,grey metal gate,open,...,NaN,Yes,<NA>,10/03/2026 09:46:11.000 GMT+08:00,2.1,0.47,37.0,1.0,1.0,raw_dataset_1


In [350]:
# Preview the first 5 rows of raw_dataset_2 to spot-check the load
df2 = processed_datasets['raw_dataset_2']
df2.head()

,feature_id,name,category,subcategory,street,barangay,latitude,longitude,location_description,operating_status,...,remarks,verified,establishment_pic,Time,Horizontal Accuracy,HDOP,Satellites in Use,Fix ID,Averaged Count,source_file
0,LUC_B02_025,Ven's Puto Seko,Food and Dining,Souvenir,A. Bonifacio,Barangay 2,14.1133775,121.5550775,NaN,Closed,...,NaN,Yes,<NA>,09/29/2026 21:52:28.000 GMT+08:00,1.7,0.53,34,1,6,raw_dataset_2
1,LUC_B02_026,"Bumibili Ginto at Pilak Sirang Alahas, Old coins",Financial,Pawnshop,A. Bonifacio,Barangay 2,14.11348667,121.5549017,NaN,Closed,...,NaN,Yes,<NA>,09/29/2026 21:55:56.000 GMT+08:00,1.5,0.45,41,1,6,raw_dataset_2
2,LUC_B02_027,Kimpoy Shoes Bags Repair,Automotive,Repair Shop,A. Bonifacio,Barangay 2,14.113875,121.5540717,NaN,Closed,...,NaN,Yes,<NA>,09/29/2026 22:01:03.000 GMT+08:00,1.6,0.48,38,1,9,raw_dataset_2
3,LUC_B02_028,Nicolo's Milk Tea,Food and Dining,Cafe,A. Bonifacio,Barangay 2,14.11386833,121.5539983,NaN,Closed,...,NaN,Yes,<NA>,09/29/2026 22:24:41.000 GMT+08:00,1.4,0.5,39,1,6,raw_dataset_2
4,LUC_B02_029,Frosty Hail - Purified Tube Ice,Retail,Other,A. Bonifacio,Barangay 2,14.11385833,121.55396,NaN,Closed,...,NaN,Yes,<NA>,09/29/2026 22:28:40.000 GMT+08:00,1.7,0.48,39,1,11,raw_dataset_2


In [351]:
# Preview the first 5 rows of raw_dataset_3 to spot-check the load
df3 = processed_datasets['raw_dataset_3']
df3.head()

,feature_id,name,category,subcategory,street,barangay,latitude,longitude,location_description,operating_status,...,remarks,verified,establishment_pic,Time,Horizontal Accuracy,HDOP,Satellites in Use,Fix ID,Averaged Count,source_file
0,LUC_B09_001,Agee's Food Hub,Food and Dining,Restaurant,La Purisima Conception,Barangay 9,14.11479441,121.5559836,First floor,Open,...,NaN,Yes,<NA>,10/02/2026 18:37:02.000 GMT+08:00,3.79,0.5,36,1,5,raw_dataset_3
1,LUC_B09_002,The Lemon LAB PH,Food and Dining,Restaurant,Don V. Cadelina,Barangay 9,14.11491013,121.5561251,NaN,Open,...,NaN,Yes,<NA>,10/02/2026 18:43:30.000 GMT+08:00,9.935,0.5,35,1,5,raw_dataset_3
2,LUC_B09_003,LHCN Fashion Botique,Retail,Clothing,Don V. Cadelina,Barangay 9,14.11503451,121.5560581,First floor,Open,...,NaN,Yes,<NA>,10/02/2026 18:47:37.000 GMT+08:00,3.79,0.5,35,1,6,raw_dataset_3
3,LUC_B09_004,Ropharisa Photo Image,Professional Services,Printing Shop,Regidor,Barangay 9,14.11512294,121.5557111,Second floor,Open,...,NaN,Yes,<NA>,10/02/2026 18:54:49.000 GMT+08:00,9.935,0.6,36,1,6,raw_dataset_3
4,LUC_B09_005,Dahilig Live Tilapia,Other,Other,Regidor,Barangay 9,14.1151541,121.555698,first floor,closed,...,NaN,Yes,<NA>,10/02/2026 18:57:04.000 GMT+08:00,3.79,0.4,38,1,6,raw_dataset_3


In [352]:
# Preview the first 5 rows of raw_dataset_4 to spot-check the load
df4 = processed_datasets['raw_dataset_4']
df4.head()

,feature_id,name,category,subcategory,street,barangay,latitude,longitude,location_description,operating_status,...,remarks,verified,establishment_pic,Time,Horizontal Accuracy,HDOP,Satellites in Use,Fix ID,Averaged Count,source_file
0,LUC_B08_011,Abcede's Resto,Food and Dining,Restaurant,Quezon Avenue,Barangay 8,14.11363157,121.5562345,NaN,Open,...,NaN,Yes,<NA>,10/01/2026 14:45:52.000 GMT+08:00,3,0.4,37,1,49,raw_dataset_4
1,LUC_B08_012,Generica drugstore,Health,Pharmacy,Quezon Avenue,Barangay 8,14.11374222,121.5561989,NaN,Open,...,NaN,Yes,<NA>,10/01/2026 14:50:34.000 GMT+08:00,2.5,0.4,36,1,3,raw_dataset_4
2,LUC_B08_013,Lola Pelang's,Food and Dining,Eatery,Quezon Avenue,Barangay 8,14.113745,121.55615,NaN,Open,...,NaN,Yes,<NA>,10/01/2026 14:55:57.000 GMT+08:00,2.5,0.4,34,1,1,raw_dataset_4
3,LUC_B08_014,Dadbod,Food and Dining,Restaurant,Quezon Avenue,Barangay 8,14.11382333,121.5559267,NaN,Closed,...,NaN,Yes,<NA>,10/01/2026 14:58:47.000 GMT+08:00,3,0.4,32,1,1,raw_dataset_4
4,LUC_B08_016,Mang Inasal,Food and Dining,Fast Food,Quezon Avenue,Barangay 8,14.11389372,121.5555892,NaN,Open,...,NaN,Yes,<NA>,10/01/2026 15:07:34.000 GMT+08:00,3,0.4,38,1,13,raw_dataset_4


In [353]:
# Preview the first 5 rows of raw_dataset_5 to spot-check the load
df5 = processed_datasets['raw_dataset_5']
df5.head()

,feature_id,name,category,subcategory,street,barangay,latitude,longitude,location_description,operating_status,...,remarks,verified,establishment_pic,Time,Horizontal Accuracy,HDOP,Satellites in Use,Fix ID,Averaged Count,source_file
0,LUC_BO5_001,Lote Fitness Gym,Food and Dining,Restaurant,Fidel Rada St.,Barangay 5,14.112805,121.55589,Near Multi Purpose hall,Open,...,Near Multi purpose hall,Yes,<NA>,10/03/2026 08:58:42.000 GMT+08:00,11.2,0.56,37,1,10,raw_dataset_5
1,LUC_B05_002,Brgy. 5 Multi-Purpose Hall,Government,Barangay Office,Balintawak St.,Barangay 1,14.11238667,121.5556533,Dulo ng balintawak street,Open,...,NaN,Yes,<NA>,10/03/2026 09:04:16.000 GMT+08:00,12.3,0.53,33,1,10,raw_dataset_5
2,LUC_B05_003,Pedron Dental Clinic,Health,Dental Clinic,NaN,Barangay 1,14.112435,121.555725,near brgy 5 multi-purpose hall,open,...,NaN,Yes,<NA>,10/03/2026 09:10:40.000 GMT+08:00,10.7,0.48,37,1,9,raw_dataset_5
3,LUC_B05_004,SK Hall,Government,Barangay Office,Balintawak St.,Barangay 1,14.11257667,121.5557483,near brgy. hall,close,...,NaN,Yes,<NA>,10/03/2026 09:18:38.000 GMT+08:00,11.5,0.51,31,1,8,raw_dataset_5
4,LUC_B05_005,Mikay Sari-Sari Store,Food and Dining,Restaurant,Balintawak st.,Barangay 1,14.11266111,121.5558833,Near Lote fitness gym and Multi-purpose hall,open,...,NaN,Yes,<NA>,10/03/2026 09:39:21.000 GMT+08:00,2.4,0.6,34,1,9,raw_dataset_5


In [354]:
# Preview the first 5 rows of raw_dataset_6 to spot-check the load
df6 = processed_datasets['raw_dataset_6']
df6.head()

,feature_id,name,category,subcategory,street,barangay,latitude,longitude,location_description,operating_status,...,remarks,verified,establishment_pic,Time,Horizontal Accuracy,HDOP,Satellites in Use,Fix ID,Averaged Count,source_file
0,LUC_B04_001,ZUREA Hotel,Accommodation,Hotel,Conception,Barangay 4,14.11624683,121.5525789,NaN,Open,...,NaN,Yes,<NA>,09/29/2026 18:07:25.000 GMT+08:00,3.79,0.5,36,1,10,raw_dataset_6
1,LUC_B04_002,NaN,Retail,Other,Conception,Barangay 4,14.11629414,121.5523815,NaN,open,...,NaN,Yes,<NA>,09/29/2026 18:11:12.000 GMT+08:00,3.79,0.7,25,1,189,raw_dataset_6
2,LUC_B04_003,NaN,Retail,Sari Sari Store,Conception,Barangay 4,14.11641405,121.5521382,NaN,Open,...,NaN,Yes,<NA>,09/29/2026 18:13:37.000 GMT+08:00,3.79,1.5,23,1,126,raw_dataset_6
3,LUC_B04_004,NaN,Personal Services,Barbershop,Conception,Barangay 4,14.11644609,121.5520527,NaN,Open,...,NaN,Yes,<NA>,09/29/2026 18:15:40.000 GMT+08:00,3.79,1.4,31,1,123,raw_dataset_6
4,LUC_B04_005,Irene's Sewing Shop,Personal Services,Other,Conception,Barangay 4,14.11646808,121.5520098,Near barbershop,open,...,NaN,Yes,<NA>,09/29/2026 18:18:05.000 GMT+08:00,9.935,0.8,29,1,7,raw_dataset_6


In [355]:
# Preview the first 5 rows of raw_dataset_7 to spot-check the load
df7 = processed_datasets['raw_dataset_7']
df7.head()

,feature_id,name,category,subcategory,street,barangay,latitude,longitude,location_description,operating_status,...,remarks,verified,establishment_pic,Time,Horizontal Accuracy,HDOP,Satellites in Use,Fix ID,Averaged Count,source_file
0,LUC_B02_001,NaN,Accommodation,Boarding House,A. Racelis,Barangay 2,14.11247803,121.5577814,NaN,NaN,...,NaN,Yes,<NA>,09/28/2026 20:05:30.000 GMT+08:00,3.79,0.5,24,1,1,raw_dataset_7
1,LUC_B02_002,Abustan Funeral Parlor,Professional Services,Office,A. Bonifacio,Barangay 2,14.11246023,121.5574515,Along Kamatian River,Closed,...,NaN,Yes,<NA>,09/28/2026 20:11:50.000 GMT+08:00,3.79,0.5,33,1,10,raw_dataset_7
2,LUC_B02_003,Lhoida Sari Sari Store,Retail,Sari Sari Store,G. Cadeliña,Barangay 2,14.1124061,121.5573801,Alongside Abustan Funeral Parlor,Open,...,NaN,Yes,<NA>,09/28/2026 20:18:26.000 GMT+08:00,3.79,0.4,37,1,1,raw_dataset_7
3,LUC_B02_004,Tan-Tan,Automotive,Repair Shop,A. Bonifacio,Barangay 2,14.11244611,121.5572275,NaN,Closed,...,NaN,Yes,<NA>,09/28/2026 20:23:21.000 GMT+08:00,9.935,0.4,37,1,20,raw_dataset_7
4,LUC_B02_005,VMCO Sari Sari Store,Retail,Sari Sari Store,A. Bonifacio,Barangay 2,14.11259878,121.556882,NaN,Open,...,NaN,Yes,<NA>,09/28/2026 20:29:54.000 GMT+08:00,3.79,0.4,34,1,8,raw_dataset_7


In [356]:
# Preview the first 5 rows of raw_dataset_8 to spot-check the load
df8 = processed_datasets['raw_dataset_8']
df8.head()

,feature_id,name,category,subcategory,street,barangay,latitude,longitude,location_description,operating_status,...,remarks,verified,establishment_pic,Time,Horizontal Accuracy,HDOP,Satellites in Use,Fix ID,Averaged Count,source_file
0,LUC_B02_019,Balmeo's Special Longganisang Lucban,Retail,Other,A. Bonifacio Street,Barangay 2,14.11358325,121.5546189,NaN,Open,...,NaN,Yes,<NA>,09/28/2026 21:17:57.000 GMT+08:00,9.935,0.6,31,1,1,raw_dataset_8
1,LUC_B02_020,Shower Of Blessing,Retail,Grocery,NaN,Barangay 2,14.11376617,121.5543987,NaN,Closed,...,NaN,Yes,<NA>,09/28/2026 21:21:34.000 GMT+08:00,3.79,0.5,34,1,12,raw_dataset_8
2,LUC_B02_021,CARD MBA,Financial,Bank,A. Bonifacio,Barangay 2,14.11365513,121.5543662,NaN,Closed,...,NaN,Yes,<NA>,09/28/2026 21:24:20.000 GMT+08:00,3.79,0.5,38,1,11,raw_dataset_8
3,LUC_B02_022,Mervin and Vivian Motor Parts And Repair,Automotive,Repair Shop,A. Bonifacio,Barangay 2,14.11373485,121.5543044,NaN,Closed,...,NaN,Yes,<NA>,09/28/2026 21:27:07.000 GMT+08:00,9.935,0.5,36,1,13,raw_dataset_8
4,LUC_B02_023,Jun Balmeo Store,Retail,Grocery,A. Bonifacio,Barangay 2,14.11377369,121.5541041,NaN,Closed,...,Long standing grocery for sari sari store,Yes,<NA>,09/28/2026 21:30:30.000 GMT+08:00,9.935,0.5,35,1,33,raw_dataset_8


In [357]:
# Preview the first 5 rows of raw_dataset_9 to spot-check the load
df9 = processed_datasets['raw_dataset_9']
df9.head()

,feature_id,name,category,subcategory,street,barangay,latitude,longitude,location_description,operating_status,...,remarks,verified,establishment_pic,Time,Horizontal Accuracy,HDOP,Satellites in Use,Fix ID,Averaged Count,source_file
0,LUC_B03_001,Hazzie Store,Retail,Sari-sari Store,Hobart Dator Street,Barangay 3,14.114485556,121.550593333,NaN,Open,...,NaN,Yes,CPE15-Midterm-Project_20261003_160402533.jpg,10/03/2026 16:02:52.000 GMT+08:00,2.7,1.06,19.0,1.0,3.0,raw_dataset_9
1,LUC_B03_002,Ning's Store,Retail,Sari-sari Store,Hobart Dator Street,Barangay 3,14.114090354,121.550054513,NaN,Open,...,NaN,Yes,CPE15-Midterm-Project_20261003_160827858.jpg,10/03/2026 16:08:22.000 GMT+08:00,1.7,0.47,36.0,0.0,1.0,raw_dataset_9
2,LUC_B03_003,Cabigan Mini Store,Retail,Sari-sari Store,Hobart Dator Street,Barangay 3,14.114839834,121.551048607,NaN,Open,...,NaN,Yes,CPE15-Midterm-Project_20261003_161445426.jpg,10/03/2026 16:13:52.000 GMT+08:00,1.7,0.49,33.0,0.0,1.0,raw_dataset_9
3,LUC_B03_004,VM Sun,Retail,Clothing,A. Bonifacio,Barangay 3,14.115182221,121.551094204,NaN,Open,...,NaN,Yes,CPE15-Midterm-Project_20261003_161854354.jpg,10/03/2026 16:18:27.000 GMT+08:00,1.7,0.5,31.0,0.0,1.0,raw_dataset_9
4,LUC_B03_005,EKRM Store,Retail,Convenience Store,129 Cadavez St.,Barangay 3,14.115301667,121.5508,NaN,Open,...,NaN,Yes,CPE15-Midterm-Project_20261003_162119150.jpg,10/03/2026 16:20:49.000 GMT+08:00,2.0,0.51,33.0,1.0,1.0,raw_dataset_9


In [358]:
# Preview the first 5 rows of raw_dataset_10 to spot-check the load
df10 = processed_datasets['raw_dataset_10']
df10.head()

,feature_id,name,category,subcategory,street,barangay,latitude,longitude,location_description,operating_status,...,remarks,verified,establishment_pic,Time,Horizontal Accuracy,HDOP,Satellites in Use,Fix ID,Averaged Count,source_file
0,LUC_B06_001,RMT's Clinical Laboratory,Health,Clinic,Placencia Street,Barangay 6,14.115498333,121.554031667,corner of the street. connected to RRN Interac...,open,...,NaN,Yes,NaN,10/05/2026 10:55:32.000 GMT+08:00,1.2,0.41,49.0,1.0,6.0,raw_dataset_10
1,LUC_B06_002,RRN Integrative Health Clinic,Health,Clinic,Placencia Street,Barangay 6,14.115498333,121.554031667,Corner of the street. Connected to RMT's Clini...,open,...,NaN,Yes,NaN,10/05/2026 10:57:35.000 GMT+08:00,1.5,0.39,47.0,1.0,6.0,raw_dataset_10
2,LUC_B06_003,The Skin Health Clinic,Health,Clinic,Placencia Street,Barangay 6,14.115498333,121.554031667,Corner of the street. Connected with RMT's Cli...,open,...,Dermatology Clinic,Yes,NaN,10/05/2026 10:59:03.000 GMT+08:00,1.3,0.4,51.0,1.0,6.0,raw_dataset_10
3,LUC_B06_004,Parinas Medical Clinic,Health,Clinic,Placencia Street,Barangay 6,14.115498333,121.554031667,Corner of the street. Connected to RRN Integra...,open,...,NaN,Yes,NaN,10/05/2026 11:00:07.000 GMT+08:00,1.4,0.38,49.0,1.0,7.0,raw_dataset_10
4,LUC_B06_005,CE MARTINEZ APPAREL SHOP,Professional Sevices,Sewing Shop,La Purisima Concepcion,Barangay 6,14.11553,121.55396,NaN,open,...,NaN,Yes,NaN,10/05/2026 11:01:15.000 GMT+08:00,1.6,0.39,48.0,1.0,6.0,raw_dataset_10


## Setting up the dataframe

In [359]:
# Data Aggregation

# Stack all 10 cleaned DataFrames into a single raw master DataFrame
df_raw = pd.concat(processed_datasets.values(), ignore_index=True)

# Enforce the spec column order on the merged raw data
df_raw = df_raw[output_columns]

# Work on a copy so the aggregated raw data (df_raw) stays untouched
df = df_raw.copy()

In [360]:
# Verify all 16 required columns are present, in the spec's order
required_columns = [
    'feature_id', 'name', 'category', 'subcategory',
    'street', 'barangay', 'latitude', 'longitude',
    'location_description', 'operating_status', 'opening_hours',
    'contact_info', 'date_collected', 'collector',
    'remarks', 'verified'
]

missing_columns = [c for c in required_columns if c not in df_raw.columns]
assert not missing_columns, f"Missing required columns: {missing_columns}"

assert list(df_raw.columns[:len(required_columns)]) == required_columns, (
    "Required columns are out of spec order"
)

print("OK:", len(required_columns), "required columns present, in spec order")
print("QA/traceability columns:", list(df_raw.columns[len(required_columns):]))

OK: 16 required columns present, in spec order
QA/traceability columns: ['source_file', 'establishment_pic', 'Time', 'Horizontal Accuracy', 'HDOP', 'Satellites in Use', 'Fix ID', 'Averaged Count']


In [361]:
# Structural audit: column dtypes, non-null counts, and memory usage
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 767 entries, 0 to 766
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   feature_id            767 non-null    str           
 1   name                  744 non-null    str           
 2   category              767 non-null    str           
 3   subcategory           767 non-null    str           
 4   street                758 non-null    str           
 5   barangay              767 non-null    str           
 6   latitude              767 non-null    str           
 7   longitude             767 non-null    str           
 8   location_description  99 non-null     str           
 9   operating_status      762 non-null    str           
 10  opening_hours         127 non-null    str           
 11  contact_info          142 non-null    str           
 12  date_collected        753 non-null    datetime64[us]
 13  collector             765 non-n

In [362]:
# Summary audit: shape, missing values, duplicate rows, and duplicate feature IDs
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nUnique feature IDs:")
print(df['feature_id'].nunique())

print("\nDuplicate feature IDs:")
print(df['feature_id'].duplicated().sum())

Rows: 767
Columns: 24

Missing values:
feature_id                0
name                     23
category                  0
subcategory               0
street                    9
barangay                  0
latitude                  0
longitude                 0
location_description    668
operating_status          5
opening_hours           640
contact_info            625
date_collected           14
collector                 2
remarks                 706
verified                  0
source_file               0
establishment_pic       737
Time                      0
Horizontal Accuracy       0
HDOP                      0
Satellites in Use         0
Fix ID                    0
Averaged Count            0
dtype: int64

Duplicate rows:
0

Unique feature IDs:
705

Duplicate feature IDs:
62


In [363]:
# Statistical audit: descriptive statistics per column (transposed for readability)
df.describe().T

,count,mean,min,25%,50%,75%,max
date_collected,753,2026-11-05 09:45:10.756972,2026-01-10 00:00:00,2026-05-10 00:00:00,2026-10-01 00:00:00,2026-10-01 00:00:00,2206-10-03 00:00:00


In [364]:
# Frequency of each feature_id to reveal duplicated or repeated records
df['feature_id'].value_counts()

feature_id
LUC_B04_026    6
LUC_B04_003    3
LUC_B04_009    3
LUC_B04_012    3
LUC_B04_016    3
              ..
LUC_B06_064    1
LUC_B06_065    1
LUC_B06_066    1
LUC_B06_067    1
LUC_B06_068    1
Name: count, Length: 705, dtype: int64

In [365]:
# Compare unique feature IDs against total entries to gauge duplication
print("feature_id:", df['feature_id'].nunique(), "\ntotal entries:", len(df))

feature_id: 705 
total entries: 767


In [366]:
# Frequency of establishment names to spot possible repeat entries
df['name'].value_counts()

name
Lucky Bookstore                 3
Ven's Puto Seko                 3
Colar                           3
Boss Tabi Inihaw                2
Shiela's Frozen Product         2
                               ..
Roland's Lambanog               1
Petshop                         1
BiggieTalk Global Department    1
Myori's                         1
Deveza Farm                     1
Name: count, Length: 722, dtype: int64

In [367]:
# Distribution of records across barangays
df['barangay'].value_counts()

barangay
Barangay 8     162
Barangay 9     126
Barangay 2     114
Barangay 10    103
Barangay 4     102
Barangay 6      68
Barangay 5      32
Barangay 1      30
Barangay 3      30
Name: count, dtype: int64

In [368]:
# Top 15 data collectors by number of records
df['collector'].value_counts().head(15)

collector
ERINCO, Raphael Joshua V.           55
ARELLANO, Arvin L.                  48
Cadao, Janusz Gabriel S.            39
APOSTOL, Janelle Mae A.             38
Escobal, Gwyneth A.                 34
Reginal, Reynel M.                  31
Salumbides, Emanuel A.              29
Libao, Ried Carlo C.                29
PALMA, Mark, M.                     24
Javin, Mark Justine P.              22
ABRACIA, JESHEILA ANGELA J.         21
RAMOS, Isaih Thomas, E.             21
De Chavez, Sharaine Princess, P.    21
Alfaro, Justin Alexandre, B.        21
Carable, Jillian Mae, V.            19
Name: count, dtype: int64

In [369]:
# Distribution of category values (first pass)
df['category'].value_counts()

category
Retail                   288
Food and Dining          187
Personal Services         65
Other                     49
Professional Services     43
Health                    43
Financial                 30
Accommodation             18
Automotive                15
Government                11
Professional Sevices       9
Tourism                    8
Financial Services         1
Name: count, dtype: int64

In [370]:
# Distribution of category values (repeat of the previous cell)
df['category'].value_counts()

category
Retail                   288
Food and Dining          187
Personal Services         65
Other                     49
Professional Services     43
Health                    43
Financial                 30
Accommodation             18
Automotive                15
Government                11
Professional Sevices       9
Tourism                    8
Financial Services         1
Name: count, dtype: int64

In [371]:
# Distribution of subcategory values
df['subcategory'].value_counts()

subcategory
Other                 134
Sari Sari Store       112
Eatery                 56
Restaurant             44
Grocery                37
                     ... 
CDC                     1
Transport Station       1
Pet Shop                1
Teaching Services       1
Tourist Attraction      1
Name: count, Length: 61, dtype: int64

In [372]:
# Distribution of operating_status values to expose casing and typo variants
df['operating_status'].value_counts()

operating_status
Open                  448
open                  202
Closed                 56
CLOSED                 36
closed                  6
close                   4
OPEN                    4
Close                   2
closed on Saturday      1
Oepn                    1
closel                  1
8AM - 2AM               1
Name: count, dtype: int64

In [373]:
# Top 20 streets by number of records
df['street'].value_counts().head(20)

street
Quezon Avenue             101
A. Racelis                 64
San Luis                   53
Lucban-Sampaloc Road       37
A. Bonifacio               32
La Purisima Concepcion     30
Katipunan                  28
Regidor                    27
Fidel Rada                 21
Don V. Cadelina            17
Gomburza Street            17
General Lukban Street      14
Bonifacio                  14
Gomburza                   13
Gen. Lukban                12
Quezon avenue              11
La Purisma Concepcion      11
Rizal Avenue               11
Lopez Jaena                10
Racelis                    10
Name: count, dtype: int64

In [374]:
# Count of verification flags across the dataset
df['verified'].value_counts()

verified
Yes    767
Name: count, dtype: int64

In [375]:
# Most frequent latitude values to spot repeated or default coordinates
df['latitude'].value_counts().head(10)

latitude
14.115498333    4
14.112825       3
14.112738333    2
14.113256667    2
14.113625       2
14.113183333    2
14.113381667    2
14.11451833     2
14.115578333    2
14.116286389    2
Name: count, dtype: int64

In [376]:
# Most frequent longitude values to spot repeated or default coordinates
df['longitude'].value_counts().head(10)

longitude
121.554031667    4
121.557915       3
121.55746        2
121.557573333    2
121.557526667    2
121.5572         2
121.55396        2
121.5532483      2
121.5558042      2
121.5560217      2
Name: count, dtype: int64

In [377]:
# Count rows that share identical (latitude, longitude) pairs
# (compare as numeric values so differently formatted strings still match)
coords_numeric = pd.DataFrame({
    'latitude': pd.to_numeric(df['latitude'], errors='coerce'),
    'longitude': pd.to_numeric(df['longitude'], errors='coerce')
})

coords_numeric.duplicated().value_counts()

False    756
True      11
Name: count, dtype: int64

In [378]:
# Inspect every record assigned to 'Barangay 1'
df[df['barangay'] == 'Barangay 1']

,feature_id,name,category,subcategory,street,barangay,latitude,longitude,location_description,operating_status,...,remarks,verified,source_file,establishment_pic,Time,Horizontal Accuracy,HDOP,Satellites in Use,Fix ID,Averaged Count
103,LUC_B01_002,Osolle Electrical Supply,Retail,Electronics,La Purisma Concepcion,Barangay 1,14.11401,121.557813333,NaN,open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 15:09:48.000 GMT+08:00,1.6,0.46,39.0,1.0,4.0
104,LUC_B01_003,Rkov Motorcle Parts,Automotive,Parts Shop,La Purisma Concepcion,Barangay 1,14.11416,121.55741,NaN,open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 15:11:35.000 GMT+08:00,1.7,0.45,38.0,1.0,4.0
105,LUC_B01_004,Yssa's Sari-sari Store,Retail,Sari Sari Store,La Purisma Concepcion,Barangay 1,14.1142175,121.557288333,NaN,open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 15:12:59.000 GMT+08:00,1.6,0.5,40.0,1.0,4.0
106,LUC_B01_005,Purr J,Retail,Other,La Purisma Concepcion,Barangay 1,14.114335333,121.557111667,NaN,open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 15:16:56.000 GMT+08:00,1.5,0.43,40.0,1.0,5.0
107,LUC_B01_006,Flora's Sari-sari Store,Retail,Sari Sari Store,La Purisma Concepcion,Barangay 1,14.114241667,121.557008333,NaN,open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 15:18:48.000 GMT+08:00,1.7,0.44,41.0,1.0,4.0
108,LUC_B01_007,Kabasi's Sari-sari Store,Retail,Sari Sari Store,La Purisma Concepcion,Barangay 1,14.114413333,121.55688875,NaN,closed,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 15:19:47.000 GMT+08:00,1.7,0.51,33.0,1.0,4.0
109,LUC_B01_008,Rosemarie Store,Retail,Sari Sari Store,La Purisma Concepcion,Barangay 1,14.114548095,121.556532619,NaN,open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 15:21:33.000 GMT+08:00,1.6,0.46,40.0,1.0,7.0
110,LUC_B01_009,Good Shepherd Diocesan School-IFI,Other,Other,La Purisma Concepcion,Barangay 1,14.114551458,121.556408333,NaN,open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 15:22:58.000 GMT+08:00,1.5,0.44,42.0,1.0,8.0
111,LUC_B01_010,Kenby Pet Supplies,Retail,Other,La Purisma Concepcion,Barangay 1,14.114696667,121.55616,NaN,open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 15:24:01.000 GMT+08:00,1.6,0.45,40.0,1.0,4.0
112,LUC_B01_011,Chicks and Babes,Food and Dining,Restaurant,La Purisma Concepcion,Barangay 1,14.114675,121.556066667,NaN,open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 15:25:00.000 GMT+08:00,1.6,0.51,37.0,1.0,4.0


## Gerenal Data Cleaning and Handling Missing Values

The Structure Audit: See data types and missing value counts

df.info()

The Statistical Audit: See averages, min, max (spots wild outliers instantly)

df.describe()

The Emptiness Audit: Count exactly how many NaN values sit in each column

df.isnull().sum()

The Duplication Audit: Check if rows are accidentally duplicated

df.duplicated().sum()

In [379]:
# .between() can be used for datetime to check if the dates are within the certain range

In [380]:
# Rank columns by number of missing values (worst first)
missing = df.isnull().sum().to_frame('n_missing')
missing.sort_values('n_missing', ascending=False)

,n_missing
establishment_pic,737
remarks,706
location_description,668
opening_hours,640
contact_info,625
name,23
date_collected,14
street,9
operating_status,5
collector,2


In [381]:
# Isolate every row whose feature_id appears more than once
duplicate_ids = df[df['feature_id'].duplicated(keep=False)]

# Sort so each duplicate group sits together for side-by-side comparison
duplicate_ids = duplicate_ids.sort_values('feature_id')

duplicate_ids

,feature_id,name,category,subcategory,street,barangay,latitude,longitude,location_description,operating_status,...,remarks,verified,source_file,establishment_pic,Time,Horizontal Accuracy,HDOP,Satellites in Use,Fix ID,Averaged Count
540,LUC_B02_059,Kantuhan Food House,Food and Dining,Eatery,Don V. Cadelina,Barangay 2,14.11309936,121.5551719,NaN,Open,...,NaN,Yes,raw_dataset_7,<NA>,09/30/2026 11:17:52.000 GMT+08:00,3.79,0.4,34,1,22
541,LUC_B02_059,Bahay Pamahalaan Barangay 2,Government,Barangay Office,Don V. Cadelina,Barangay 2,14.11313646,121.555209,NaN,Open,...,NaN,Yes,raw_dataset_7,<NA>,09/30/2026 11:19:34.000 GMT+08:00,3.79,0.4,34,1,102
694,LUC_B03_025,Trendee by Dee Jay Abustan,Personal Services,Fashion Design,Armando Racelis Avenue,Barangay 3,14.115592564,121.551067047,NaN,Open,...,NaN,Yes,raw_dataset_9,CPE15-Midterm-Project_20261003_171741219.jpg,10/03/2026 17:16:54.000 GMT+08:00,1.9,0.61,29.0,0.0,1.0
693,LUC_B03_025,Jepoy Sisig,Food and Dining,Eatery,Armando Racelis Avenue,Barangay 3,14.115499245,121.551266536,NaN,Open,...,NaN,Yes,raw_dataset_9,CPE15-Midterm-Project_20261003_171425690.jpg,10/03/2026 17:13:47.000 GMT+08:00,1.8,0.47,35.0,0.0,2.0
604,LUC_B04_001,ZUREA Hotel,Accommodation,Hotel,Conception Street,Barangay 4,14.11623334,121.5524813,NaN,OPEN,...,NaN,Yes,raw_dataset_8,<NA>,10/01/2026 22:03:05.000 GMT+08:00,32.537,4.3,4,1,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
267,LUC_B09_125,Lena's Store,Retail,Sari Sari Store,Gil Rada,Barangay 9,14.11838744,121.5575563,NaN,Open,...,NaN,Yes,raw_dataset_3,<NA>,10/03/2026 16:53:14.000 GMT+08:00,9.935,0.4,44,1,5
12,LUC_B10_012,Itlog ni KUYA,Food and Dining,Grocery,Quezon avenue,Barangay 10,14.112911111,121.558054167,NaN,open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 10:02:54.000 GMT+08:00,1.5,0.43,40.0,1.0,6.0
11,LUC_B10_012,BugMac,Food and Dining,Fast Food,Quezon avenue,Barangay 10,14.112948611,121.558051667,NaN,open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 10:01:14.000 GMT+08:00,1.7,0.44,39.0,1.0,24.0
31,LUC_B10_031,Boss Tabi Inihaw,Food and Dining,Eatery,Quezon Avenue,Barangay 10,14.113625,121.557573333,NaN,open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 10:43:09.000 GMT+08:00,1.6,0.47,41.0,1.0,3.0


In [382]:
# Isolate non-null duplicated names, sorted by name, to find repeat establishments
duplicate_names = df[
    df['name'].notna() &
    df['name'].duplicated(keep=False)
].sort_values('name')

duplicate_names

,feature_id,name,category,subcategory,street,barangay,latitude,longitude,location_description,operating_status,...,remarks,verified,source_file,establishment_pic,Time,Horizontal Accuracy,HDOP,Satellites in Use,Fix ID,Averaged Count
270,LUC_B08_011,Abcede's Resto,Food and Dining,Restaurant,Quezon Avenue,Barangay 8,14.11363157,121.5562345,NaN,Open,...,NaN,Yes,raw_dataset_4,<NA>,10/01/2026 14:45:52.000 GMT+08:00,3,0.4,37,1,49
101,LUC_B10_102,Abcede's Resto,Food and Dining,Restaurant,Quezon Avenue,Barangay 10,14.113603333,121.556273333,NaN,open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 14:57:33.000 GMT+08:00,1.7,0.48,40.0,1.0,4.0
144,LUC_B09_001,Agee's Food Hub,Food and Dining,Restaurant,La Purisima Conception,Barangay 9,14.11479441,121.5559836,First floor,Open,...,NaN,Yes,raw_dataset_3,<NA>,10/02/2026 18:37:02.000 GMT+08:00,3.79,0.5,36,1,5
113,LUC_B01_012,Agee's Food Hub,Food and Dining,Eatery,La Purisma Concepcion,Barangay 1,14.114732917,121.555983333,NaN,open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 15:26:25.000 GMT+08:00,1.5,0.47,39.0,1.0,4.0
30,LUC_B10_031,Boss Tabi Inihaw,Food and Dining,Eatery,Quezon Avenue,Barangay 10,14.113625,121.557573333,NaN,open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 10:43:06.000 GMT+08:00,1.7,0.48,41.0,1.0,5.0
31,LUC_B10_031,Boss Tabi Inihaw,Food and Dining,Eatery,Quezon Avenue,Barangay 10,14.113625,121.557573333,NaN,open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 10:43:09.000 GMT+08:00,1.6,0.47,41.0,1.0,3.0
419,LUC_B08_009,Cebuana Lhuillier,Financial,Pawnshop,A. Racelis,Barangay 8,14.11363807,121.5554322,NaN,Open,...,NaN,Yes,raw_dataset_6,<NA>,10/01/2026 14:59:37.000 GMT+08:00,9.935,1,26,1,100
172,LUC_B09_029,Cebuana Lhuillier,Financial,Remittance Center,Regidor,Barangay 9,14.11531584,121.555777,NaN,Open,...,NaN,Yes,raw_dataset_3,<NA>,10/02/2026 19:51:30.000 GMT+08:00,9.935,0.6,35,1,5
163,LUC_B09_020,Colar,Retail,Clothing,Lopez Jaena,Barangay 9,14.11542207,121.5552589,NaN,Open,...,NaN,Yes,raw_dataset_3,<NA>,10/02/2026 19:27:55.000 GMT+08:00,3.79,0.5,38,1,4
162,LUC_B09_019,Colar,Retail,Clothing,Lopez jaena,Barangay 9,14.11539792,121.5552466,NaN,Open,...,NaN,Yes,raw_dataset_3,<NA>,10/02/2026 19:24:10.000 GMT+08:00,9.935,0.5,36,1,4


In [383]:
# Isolate rows with duplicate coordinate pairs, sorted by coordinates
# (match on numeric values, display the original coordinate strings)
coords_numeric = pd.DataFrame({
    'latitude': pd.to_numeric(df['latitude'], errors='coerce'),
    'longitude': pd.to_numeric(df['longitude'], errors='coerce')
})

duplicate_coordinates = df[
    coords_numeric.duplicated(keep=False)
].sort_values(['latitude', 'longitude'])

duplicate_coordinates

,feature_id,name,category,subcategory,street,barangay,latitude,longitude,location_description,operating_status,...,remarks,verified,source_file,establishment_pic,Time,Horizontal Accuracy,HDOP,Satellites in Use,Fix ID,Averaged Count
0,LUC_B10_001,"mommy's tacos, burgers and pizzas",Food and Dining,Eatery,G.Cadeliña,Barangay 10,14.112738333,121.55746,food stall at the front with an eatery inside,open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 09:32:59.000 GMT+08:00,2.0,0.53,32.0,1.0,11.0
1,LUC_B10_002,bnb laundry,Personal Services,Laundry,G.Cadeliña,Barangay 10,14.112738333,121.55746,"Next to mommy's tacos, burgers and pizzas",open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 09:34:53.000 GMT+08:00,2.1,0.5,41.0,1.0,114.0
61,LUC_B10_062,Kuya Richard,Retail,Sari Sari Store,Quezon Avenue,Barangay 10,14.112825,121.557915,NaN,open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 11:48:40.000 GMT+08:00,1.7,0.45,37.0,1.0,1.0
62,LUC_B10_063,Cozy Travelers,Professional Services,Lodge,Quezon Avenue,Barangay 10,14.112825,121.557915,NaN,open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 11:49:47.000 GMT+08:00,2.3,0.5,32.0,1.0,1.0
63,LUC_B10_064,Lap Sign Arts,Professional Services,Printing Shop,Quezon Avenue,Barangay 10,14.112825,121.557915,NaN,open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 11:50:41.000 GMT+08:00,1.8,0.46,36.0,1.0,1.0
76,LUC_B10_077,The Thrifters Ave Fashion Shop,Retail,Clothing,Quezon Avenue,Barangay 10,14.113183333,121.5572,tapat ng ongville,open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 14:24:11.000 GMT+08:00,2.8,0.79,20.0,1.0,5.0
77,LUC_B10_078,Marella,Other,Other,Quezon Avenue,Barangay 10,14.113183333,121.5572,tapat ng ongville,open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 14:25:10.000 GMT+08:00,1.8,0.55,32.0,1.0,3.0
30,LUC_B10_031,Boss Tabi Inihaw,Food and Dining,Eatery,Quezon Avenue,Barangay 10,14.113625,121.557573333,NaN,open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 10:43:06.000 GMT+08:00,1.7,0.48,41.0,1.0,5.0
31,LUC_B10_031,Boss Tabi Inihaw,Food and Dining,Eatery,Quezon Avenue,Barangay 10,14.113625,121.557573333,NaN,open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 10:43:09.000 GMT+08:00,1.6,0.47,41.0,1.0,3.0
142,LUC_B02_048,GOGO SHAMAN LENDING CORPORATION,Financial,Remittance Center,A. Racelis,Barangay 2,14.11451833,121.5532483,Second Floor,Closed,...,NaN,Yes,raw_dataset_2,<NA>,09/29/2026 23:33:31.000 GMT+08:00,1.5,0.43,39,1,2


In [384]:
# Print latitude/longitude ranges to spot coordinates outside expected bounds
# (coordinates are read as text at load, so cast to numeric before comparing)
lat = pd.to_numeric(df['latitude'], errors='coerce')
lon = pd.to_numeric(df['longitude'], errors='coerce')

print(lat.min())
print(lat.max())

print(lon.min())
print(lon.max())

14.11230458
14.11881973
121.550036408
121.558576667


In [385]:
# Filter rows with missing or out-of-range coordinates (lat 0-90, lon -180 to 180)
lat = pd.to_numeric(df['latitude'], errors='coerce')
lon = pd.to_numeric(df['longitude'], errors='coerce')

bad_coordinates = df[
    lat.isna() |
    lon.isna() |
    (lat < 0) |
    (lat > 90) |
    (lon < -180) |
    (lon > 180)
]

bad_coordinates

,feature_id,name,category,subcategory,street,barangay,latitude,longitude,location_description,operating_status,...,remarks,verified,source_file,establishment_pic,Time,Horizontal Accuracy,HDOP,Satellites in Use,Fix ID,Averaged Count


In [386]:
# Correct the misspelled category label 'Professional Sevices'
category_fixes = {
    'Professional Sevices': 'Professional Services'
}

df['category'] = df['category'].replace(category_fixes)
df['category']

0            Food and Dining
1          Personal Services
2                     Retail
3                      Other
4            Food and Dining
               ...          
762                   Retail
763                    Other
764    Professional Services
765          Food and Dining
766                  Tourism
Name: category, Length: 767, dtype: str

In [387]:
# Normalize operating_status casing and typos into canonical 'Open'/'Closed'
status_fixes = {
    'open': 'Open',
    'OPEN': 'Open',
    'Oepn': 'Open',
    'close': 'Closed',
    'Close': 'Closed',
    'CLOSED': 'Closed',
    'closed': 'Closed',
    'closel': 'Closed'
}

df['operating_status'] = df['operating_status'].replace(status_fixes)
df['operating_status']

0      Open
1      Open
2      Open
3      Open
4      Open
       ... 
762    Open
763    Open
764    Open
765    Open
766    Open
Name: operating_status, Length: 767, dtype: str

In [388]:
# Normalize 'Sari-Sari Store' spelling variants into one canonical label
subcategory_fixes = {
    'Sari Sari Store': 'Sari-Sari Store',
    'Sari-sari Store': 'Sari-Sari Store'
}

df['subcategory'] = df['subcategory'].replace(subcategory_fixes)

In [389]:
# Repair every feature_id with rule-based fixes (checklist section 2),
# then enforce the canonical pattern and report each change
original_ids = df['feature_id'].copy()
df['feature_id'] = df['feature_id'].apply(standardize_feature_id)

# Audit trail: every value the repair actually changed
changed = original_ids != df['feature_id']
repaired_ids = pd.DataFrame({
    'original': original_ids[changed],
    'repaired': df['feature_id'][changed]
})
print(f"{len(repaired_ids)} malformed feature_id(s) repaired:")
print(repaired_ids.to_string())

# No ID may remain outside the LUC_BNN_NNN pattern
invalid_ids = sorted(
    df.loc[~df['feature_id'].str.match(FEATURE_ID_PATTERN), 'feature_id'].unique()
)
assert not invalid_ids, f"Malformed feature_ids after repair: {invalid_ids}"
print(f"OK: all {len(df)} feature IDs match {FEATURE_ID_PATTERN}")

8 malformed feature_id(s) repaired:
         original     repaired
284   LUC-B08_036  LUC_B08_036
334   LUC_BO5_001  LUC_B05_001
340   LUC_BO5_006  LUC_B05_006
346   LUC_BO5_011  LUC_B05_011
349   LUC_BO5_014  LUC_B05_014
612  LUC _B04_008  LUC_B04_008
620   LIC_B04_014  LUC_B04_014
628    LUC_B04_20  LUC_B04_020
OK: all 767 feature IDs match ^LUC_B\d{2}_\d{3}$


In [390]:
# Extract the barangay number encoded in feature_id (pattern LUC_B<NN>_)
df['id_barangay'] = (
    df['feature_id']
    .str.extract(r'LUC_B(\d{2})_')[0]
)

In [391]:
# Convert the extracted barangay code to numeric; failures become NaN
df['id_barangay'] = pd.to_numeric(
    df['id_barangay'],
    errors='coerce'
)

In [392]:
# Extract the barangay number written in the barangay column itself
df['actual_barangay'] = (
    df['barangay']
    .str.extract(r'(\d+)')[0]
)

# Convert to numeric; unparseable values become NaN
df['actual_barangay'] = pd.to_numeric(
    df['actual_barangay'],
    errors='coerce'
)

In [393]:
# Flag rows where the feature_id barangay code disagrees with the barangay column
id_barangay_mismatch = df[
    df['id_barangay'] != df['actual_barangay']
]

id_barangay_mismatch

,feature_id,name,category,subcategory,street,barangay,latitude,longitude,location_description,operating_status,...,source_file,establishment_pic,Time,Horizontal Accuracy,HDOP,Satellites in Use,Fix ID,Averaged Count,id_barangay,actual_barangay
102,LUC_B01_001,Mr. Yap Fresh Buko,Other,Other,Quezon Avenue,Barangay 10,14.113917667,121.558059333,NaN,Open,...,raw_dataset_1,<NA>,10/03/2026 15:06:53.000 GMT+08:00,1.6,0.43,41.0,1.0,5.0,1,10
284,LUC_B08_036,SJS,Retail,Other,Quezon Avenue,Barangay 1,14.11438333,121.5546797,NaN,Open,...,raw_dataset_4,<NA>,10/01/2026 15:33:44.000 GMT+08:00,3,0.4,37,1,5,8,1
335,LUC_B05_002,Brgy. 5 Multi-Purpose Hall,Government,Barangay Office,Balintawak St.,Barangay 1,14.11238667,121.5556533,Dulo ng balintawak street,Open,...,raw_dataset_5,<NA>,10/03/2026 09:04:16.000 GMT+08:00,12.3,0.53,33,1,10,5,1
336,LUC_B05_003,Pedron Dental Clinic,Health,Dental Clinic,NaN,Barangay 1,14.112435,121.555725,near brgy 5 multi-purpose hall,Open,...,raw_dataset_5,<NA>,10/03/2026 09:10:40.000 GMT+08:00,10.7,0.48,37,1,9,5,1
337,LUC_B05_004,SK Hall,Government,Barangay Office,Balintawak St.,Barangay 1,14.11257667,121.5557483,near brgy. hall,Closed,...,raw_dataset_5,<NA>,10/03/2026 09:18:38.000 GMT+08:00,11.5,0.51,31,1,8,5,1
338,LUC_B05_005,Mikay Sari-Sari Store,Food and Dining,Restaurant,Balintawak st.,Barangay 1,14.11266111,121.5558833,Near Lote fitness gym and Multi-purpose hall,Open,...,raw_dataset_5,<NA>,10/03/2026 09:39:21.000 GMT+08:00,2.4,0.6,34,1,9,5,1
357,LUC_B05_022,Nonette's Store,Retail,Sari-Sari Store,E. Zurbano Street,Barangay 1,14.11312167,121.5533867,Palola Way,Open,...,raw_dataset_5,<NA>,10/03/2026 11:39:09.000 GMT+08:00,11.6,0.48,36,2,1,5,1
362,LUC_B05_027,Pablo's Store,Retail,Sari-Sari Store,NaN,Barangay 1,14.112965,121.5545967,besides Pepet Restaurant,Open,...,raw_dataset_5,<NA>,10/03/2026 11:58:17.000 GMT+08:00,11.7,0.5,33,1,1,5,1
366,LUC_B05_031,Che 2x,Food and Dining,Restaurant,Marcos Tigla street,Barangay 1,14.11291,121.5549117,beside tapzi corner,Open,...,raw_dataset_5,<NA>,10/03/2026 12:07:24.000 GMT+08:00,13.5,0.54,31,1,1,5,1
390,LUC_B04_020,Pearl Vigil Hall Memorial Homes,Other,Other,Gomburza,Barangay 1,14.11553507,121.5528083,NaN,Open,...,raw_dataset_6,<NA>,09/29/2026 18:51:21.000 GMT+08:00,3.79,1,33,1,90,4,1


In [394]:
# Drop the temporary validation columns after the consistency check
df = df.drop(columns=['id_barangay', 'actual_barangay'])

## Manual Corrections and Value Fixing

In [395]:
# Missing-value report: count and percentage per column, sorted by severity
missing = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_percent': df.isnull().mean() * 100
})

# Order columns from most missing to least missing
missing = missing.sort_values(
    'missing_percent',
    ascending=False
)

missing

,missing_count,missing_percent
establishment_pic,737,96.088657
remarks,706,92.046936
location_description,668,87.092568
opening_hours,640,83.441982
contact_info,625,81.486310
name,23,2.998696
date_collected,14,1.825293
street,9,1.173403
operating_status,5,0.651890
collector,2,0.260756


In [396]:
# Re-parse date_collected as datetime (safety pass after concatenation)
df['date_collected'] = pd.to_datetime(
    df['date_collected'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

In [397]:
# Check date_collected range and flag implausible future dates
df['date_collected'].min()
df['date_collected'].max()

# Flag any collection date after 2026-12-31 as implausible
future_dates = df[
    df['date_collected'] > pd.Timestamp('2026-12-31')
]

future_dates

,feature_id,name,category,subcategory,street,barangay,latitude,longitude,location_description,operating_status,...,remarks,verified,source_file,establishment_pic,Time,Horizontal Accuracy,HDOP,Satellites in Use,Fix ID,Averaged Count
352,LUC_B05_017,Apo ni Luming pansitan,Food and Dining,Eatery,Marcus Tigla St,Barangay 5,14.11358667,121.5531567,front of orange house,Open,...,NaN,Yes,raw_dataset_5,<NA>,10/03/2026 11:18:51.000 GMT+08:00,12.4,0.41,41,1,1


In [398]:
# Inspect rows whose dates failed to parse (NaT after coercion)
df[df['date_collected'].isna()]

,feature_id,name,category,subcategory,street,barangay,latitude,longitude,location_description,operating_status,...,remarks,verified,source_file,establishment_pic,Time,Horizontal Accuracy,HDOP,Satellites in Use,Fix ID,Averaged Count
3,LUC_B10_004,ICE ICE BABY,Other,Other,G.Cadeliña,Barangay 10,14.112879561,121.55752386,front have a gate colored brown,Open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 09:43:21.000 GMT+08:00,1.8,0.51,41.0,1.0,19.0
4,LUC_B10_005,Zendong,Food and Dining,Eatery,G.Cadeliña,Barangay 10,14.11294,121.557613333,grey metal gate,Open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 09:46:11.000 GMT+08:00,2.1,0.47,37.0,1.0,1.0
14,LUC_B10_015,Joey's Barber Shop,Personal Services,Barbershop,NaN,Barangay 10,14.112889894,121.558176693,NaN,Open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 10:06:31.000 GMT+08:00,1.9,0.43,45.0,1.0,63.0
15,LUC_B10_016,CELLPHONE REPAIR SHOP,Personal Services,Electronics,Quezon avenue,Barangay 10,14.112800539,121.55823027,NaN,Open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 10:07:51.000 GMT+08:00,1.6,0.43,48.0,1.0,68.0
16,LUC_B10_017,OHMPERE,Professional Services,Electronics,Quezon avenue,Barangay 10,14.112792764,121.55831252,NaN,Open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 10:08:56.000 GMT+08:00,1.6,0.43,48.0,1.0,41.0
17,LUC_B10_018,COFI MINDED CAMP,Food and Dining,Cafe,Quezon avenue,Barangay 10,14.112836333,121.558181667,NaN,Open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 10:10:21.000 GMT+08:00,1.7,0.4,45.0,1.0,5.0
18,LUC_B10_019,BRGY. MULTI PURPOSE HALL,Government,Barangay Office,San Luis,Barangay 10,14.11293,121.558486667,NaN,Open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 10:13:15.000 GMT+08:00,1.9,0.44,37.0,1.0,6.0
19,LUC_B10_020,Mamet's sari sari store,Retail,Sari-Sari Store,San Luis,Barangay 10,14.1129,121.558576667,NaN,Open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 10:16:09.000 GMT+08:00,2.7,0.85,23.0,1.0,1.0
20,LUC_B10_021,Oblenida's sari sari store,Retail,Sari-Sari Store,San Luis,Barangay 10,14.112935,121.558561667,NaN,Open,...,NaN,Yes,raw_dataset_1,<NA>,10/03/2026 10:16:54.000 GMT+08:00,1.6,0.4,43.0,1.0,1.0
150,LUC_B09_007,Card Inc. Lucban Unit,Financial,Remittance Center,Regidor,Barangay 9,14.11504671,121.5556779,NaN,Open,...,NaN,Yes,raw_dataset_3,<NA>,10/02/2026 19:01:07.000 GMT+08:00,9.996,0.5,36,1,5
